 # 5 工具m

前置：加载工具

In [ ]:
from dotenv import load_dotenv
import os

# 关键：每次运行都强制覆盖系统环境变量
load_dotenv(override=True)

# 然后读取
api_key = os.getenv('DEEPSEEK_API_KEY')

In [ ]:
# Please install OpenAI SDK first: `pip3 install openai`
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get('DEEPSEEK_API_KEY'),
    base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Hello"},
    ],
    stream=False,
    reasoning_effort="high",
    extra_body={"thinking": {"type": "enabled"}}
)

print(response.choices[0].message.content)

5.1初体验，模拟获取天气，了解工具的基本组成

In [ ]:
# 1.使用tool装饰器定义工具
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"

5.1.2初体验，创建智能体，并绑定工具（用数组的方式绑定）

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 2.创建智能体，并绑定工具
agent = create_agent(
    model="	deepseek-v4-flash",
    tools=[get_weather]
)

# 3.调用Agent
response = agent.invoke(
    {"messages": [HumanMessage(content="杭州今天天气如何?")]},
)

for message in response['messages']:
    message.pretty_print()

5.2定义工具的几种形式

5.2.1智能体在工作时，需要将函数的名称、输入、作用传递给大模型，默认情况下这些信息的来源是：
- 工具名称：函数名
- 工具输入：函数入参
- 工具作用：函数的注释

In [ ]:
# 定义工具
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """计算指定数字的平方根"""
    return x ** 0.5

5.2.2当然，我们可以通过tool装饰器来覆盖上述信息：
- 通过装饰器定义工具名称

In [ ]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

5.2.2- 通过装饰器定义工具作用描述

In [ ]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

5.2.3- 通过装饰器定义工具入参约束
如果要覆盖工具的入参信息则会复杂很多，我们要借助于Pydantic或JSON约束。

例如，我们需要定义个查询天气的tool，借助于Pydantic来约束入参。
我们定义一个入参的模型，在模型中添加入参描述信息：

In [ ]:
# 例如一个查询天气的tool
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

5.2.4定义工具，使用定义的模型来约束入参：

In [ ]:
# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

工具定义好之后，调用方式与普通函数类似：

In [ ]:
# 调用数学工具
tool1.invoke({"x": 467})

# 调用查询天气工具
get_weather.invoke({"location": "杭州", "include_forecast": True})

当我们创建智能体时，可以把定义好的工具传递给智能体，将来模型就能得到工具信息，并根据情况判断是否需要调用工具，需要调用哪个工具了。

In [ ]:
from langchain.agents import create_agent

# 创建智能体，并添加工具
agent = create_agent(
    model="deepseek-chat",
    tools=[tool1, get_weather],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)

接下来，调用智能体，向其提问，模型会自动根据用户问题判断：
- 是否需要调用工具？
- 该调用哪个工具？
- 该传递那些参数？
并且在调用工具之后，根据工具执行结果给用户生成响应。

In [ ]:
# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="467的平方根是多少?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="北京和杭州接下来几天天气如何?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

如果采用stream模式的updates模式，可以看到工具调用的具体步骤

In [ ]:
for chunk in agent.stream(
    {"messages": [HumanMessage(content="467、529的平方根是多少?")]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")
        print()

# 5.3预定义工具m

LangChain中提供了很多预定义好的工具，方便我们使用，可使用的预定义工具列表可参考官网：
https://www.langchain.com.cn/docs/integrations/tools/m